# IOAI — 2026 Mock Word Segmentation (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-mock-word-segmentation/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Word Segmentation — 모범답안 (문자 임베딩 + BiLSTM)

각 글자를 임베딩하고 **양방향 LSTM** 으로 문맥을 읽어 글자별로 "여기가 하위단어의 끝인가?"를 이진분류한다
(BCE, 길이 마스크). 마지막 글자는 항상 경계로 강제한다. 평균 F1 ≈ 0.9 (분할 안 함 0.0 대비). GPU 수 분.

In [ ]:
import json, numpy as np
train = json.load(open("data/train.json"))       # {word: [0/1,...]}
test  = json.load(open("data/test.json"))         # {word: []}  (라벨 비어있음)
test_words = list(test.keys())
print("train", len(train), "| test", len(test_words))

In [ ]:
import torch, torch.nn as nn, random
from torch.nn.utils.rnn import pad_sequence
dev = "cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(0)

chars = sorted({c for w in train for c in w} | {c for w in test_words for c in w})
VOCAB = {c: i+1 for i, c in enumerate(chars)}      # 0 = padding
def enc(w): return torch.tensor([VOCAB.get(c, 0) for c in w])

class SegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(VOCAB)+1, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 128, 2, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(256, 1)
    def forward(self, x): return self.fc(self.lstm(self.emb(x))[0]).squeeze(-1)

net = SegNet().to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3)
bce = nn.BCEWithLogitsLoss(reduction="none")
items = list(train.items()); random.Random(0).shuffle(items)

def batches(items, bs=128):
    for i in range(0, len(items), bs):
        chunk = items[i:i+bs]
        xs = [enc(w) for w, _ in chunk]; ys = [torch.tensor(l, dtype=torch.float) for _, l in chunk]
        L = torch.tensor([len(x) for x in xs])
        yield pad_sequence(xs, batch_first=True).to(dev), pad_sequence(ys, batch_first=True).to(dev), L.to(dev)

for ep in range(8):
    net.train()
    for X, Y, L in batches(items):
        opt.zero_grad(); out = net(X)
        mask = (torch.arange(X.size(1), device=dev)[None] < L[:,None]).float()
        loss = (bce(out, Y) * mask).sum() / mask.sum(); loss.backward(); opt.step()
    print("epoch", ep, "loss", round(float(loss), 4))

net.eval(); sub = {}
with torch.no_grad():
    for i in range(0, len(test_words), 256):
        ws = test_words[i:i+256]; xs = [enc(w) for w in ws]
        X = pad_sequence(xs, batch_first=True).to(dev)
        pr = (torch.sigmoid(net(X)) > 0.5).int().cpu().numpy()
        for j, w in enumerate(ws):
            lab = pr[j][:len(w)].tolist(); lab[-1] = 1     # 마지막 글자는 항상 경계
            sub[w] = lab
json.dump(sub, open("submission.json","w"))
print("saved submission.json", len(sub))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)